# LFM2.5 Model Quantization (F16 → Q4_K_M)

**Platform**: Kaggle  
**Purpose**: Quantize your fixed F16 GGUF to Q4_K_M (~220MB) for production use.

**Steps**:
1. Upload F16 file to Kaggle (Add Data → Upload)
2. Run all cells in order
3. Download output file from `/kaggle/working/`

**Time**: ~5 minutes

In [ ]:
# Step 1: Clone and build llama.cpp
!git clone https://github.com/ggerganov/llama.cpp
!cd llama.cpp && make -j$(nproc)
print("✓ llama.cpp built successfully")

In [ ]:
# Step 2: Locate your uploaded F16 file
# If you uploaded as dataset, it's in /kaggle/input/<dataset-name>/
# If you added to notebook, check /kaggle/input/

import os
import shutil

# Find the uploaded file
print("Looking for lfm25_fixed_f16.gguf...\n")

# Check common locations
possible_paths = [
    "/kaggle/input/lfm25_fixed_f16.gguf",
    "/kaggle/working/lfm25_fixed_f16.gguf",
]

# Also check any datasets
if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        for file in files:
            if file.endswith(".gguf") or "lfm25" in file:
                possible_paths.append(os.path.join(root, file))

# Find the file
input_file = None
for path in possible_paths:
    if os.path.exists(path):
        input_file = path
        break

if input_file:
    print(f"✓ Found: {input_file}")
    file_size_mb = os.path.getsize(input_file) / (1024 * 1024)
    print(f"  Size: {file_size_mb:.1f}MB")
    
    # Copy to working directory if needed
    if not input_file.startswith("/kaggle/working"):
        print("  Copying to /kaggle/working/...")
        shutil.copy(input_file, "/kaggle/working/lfm25_fixed_f16.gguf")
        input_file = "/kaggle/working/lfm25_fixed_f16.gguf"
        print("  ✓ Copied")
else:
    print("✗ File not found!")
    print("\nPlease upload lfm25_fixed_f16.gguf:")
    print("1. Click 'Add Data' button (top right)")
    print("2. Choose 'Upload' → Select lfm25_fixed_f16.gguf")
    print("3. Wait for upload to complete")
    print("4. Re-run this cell")
    raise FileNotFoundError("Input file not found")

In [ ]:
# Step 3: Verify file
!ls -lh /kaggle/working/lfm25_fixed_f16.gguf
print("\n✓ Ready to quantize")

In [ ]:
# Step 4: Quantize F16 → Q4_K_M
print("Starting quantization...\n")
!./llama.cpp/llama-quantize \
    /kaggle/working/lfm25_fixed_f16.gguf \
    /kaggle/working/lfm25_fixed_Q4_K_M.gguf \
    Q4_K_M

print("\n" + "="*60)
print("✓ Quantization complete!")
print("="*60)

In [ ]:
# Step 5: Verify output
import os

output_file = "/kaggle/working/lfm25_fixed_Q4_K_M.gguf"

if os.path.exists(output_file):
    size_mb = os.path.getsize(output_file) / (1024 * 1024)
    print(f"✓ Output file created: {output_file}")
    print(f"  Size: {size_mb:.1f}MB (expected ~220MB)")
    
    if 210 <= size_mb <= 230:
        print("  ✓ Size looks correct!")
    else:
        print(f"  ⚠ Size is unexpected (should be ~220MB)")
    
    print("\n" + "="*60)
    print("NEXT STEPS:")
    print("="*60)
    print("1. Download the file:")
    print("   - Look in the Output section (right panel)")
    print("   - Or click 'Save Version' to commit output")
    print("   - File: lfm25_fixed_Q4_K_M.gguf")
    print("\n2. Test on device (optional but recommended)")
    print("\n3. Upload to HuggingFace")
    print("\n4. Update app code and rebuild")
    print("="*60)
else:
    print("✗ Output file not found!")
    print("Check the quantization logs above for errors.")

In [ ]:
# Step 6: List all output files
print("Files in /kaggle/working/:")
!ls -lh /kaggle/working/*.gguf 2>/dev/null || echo "No .gguf files found"